<a href="https://colab.research.google.com/github/laiba-razi/fedmed_group2/blob/feature%2Fmodel/FederatedLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kaggle
!pip install -q monai nibabel

In [ ]:
from google.colab import files
files.upload()

In [3]:
import os
os.makedirs("/root/.kaggle (4)", exist_ok=True)

!cp "kaggle (4).json" "/root/.kaggle (4)/"
!chmod 600 "/root/.kaggle (4)/kaggle (4).json"

In [4]:
!kaggle datasets download -d dschettler8845/brats-2021-task1

Dataset URL: https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1
License(s): copyright-authors
brats-2021-task1.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip -q brats-2021-task1.zip -d brats

replace brats/BraTS2021_00495.tar? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace brats/BraTS2021_00621.tar? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace brats/BraTS2021_Training_Data.tar? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y


In [ ]:
!ls -R brats

In [ ]:
!ls -R brats

In [8]:
!tar -xf brats/BraTS2021_Training_Data.tar -C brats/

tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now


In [9]:
import os
import glob

import monai
import nibabel as nib

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandCropByPosNegLabeld,
    ToTensord,
)

from monai.data import Dataset, DataLoader

In [10]:
data_dir = "brats"

# Filter the list to include only directories, excluding files like .DS_Store or .tar files
patients = sorted([p for p in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, p))])

train_files = []

for patient in patients:

    folder = os.path.join(data_dir, patient)

    train_files.append({

        "image":[
            os.path.join(folder, patient+"_t1.nii.gz"),
            os.path.join(folder, patient+"_t1ce.nii.gz"),
            os.path.join(folder, patient+"_t2.nii.gz"),
            os.path.join(folder, patient+"_flair.nii.gz"),
        ],

        "label":os.path.join(folder, patient+"_seg.nii.gz")

    })

In [11]:
train_files[0]

{'image': ['brats/BraTS2021_00000/BraTS2021_00000_t1.nii.gz',
  'brats/BraTS2021_00000/BraTS2021_00000_t1ce.nii.gz',
  'brats/BraTS2021_00000/BraTS2021_00000_t2.nii.gz',
  'brats/BraTS2021_00000/BraTS2021_00000_flair.nii.gz'],
 'label': 'brats/BraTS2021_00000/BraTS2021_00000_seg.nii.gz'}

In [12]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandCropByPosNegLabeld,
    ToTensord,
    MapLabelValued,
    Lambdad,
)

from monai.data import Dataset, DataLoader

train_transforms = Compose([

    LoadImaged(keys=["image","label"]),

    EnsureChannelFirstd(keys=["image","label"]),

    # Remap BraTS labels (0, 1, 2, 4) to sequential (0, 1, 2, 3)
    MapLabelValued(
        keys="label",
        orig_labels=(1, 2, 4),
        target_labels=(1, 2, 3)
    ),

    # Ensure label values are strictly within [0, 3] and are integers
    Lambdad(
        keys="label",
        func=lambda x: x.clamp(min=0, max=3).int()
    ),

    Orientationd(
        keys=["image","label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image","label"],
        pixdim=(1.0,1.0,1.0),
        mode=("bilinear","nearest")
    ),

    ScaleIntensityRanged(

        keys=["image"],

        a_min=0,
        a_max=3000,

        b_min=0,
        b_max=1,

        clip=True

    ),

    CropForegroundd(
        keys=["image","label"],
        source_key="image"
    ),

    ToTensord(
        keys=["image","label"]
    )

])

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [13]:
# Re-running train_transforms after fix
pass

In [14]:
# Re-running model definition after changes
pass

In [15]:
# Re-running loss and optimizer definition after changes
pass

In [16]:
train_ds = Dataset(
    data=train_files,
    transform=train_transforms
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True
)

In [17]:
sample = train_ds[0]

print(sample["image"].shape)
print(sample["label"].shape)

torch.Size([4, 136, 171, 146])
torch.Size([1, 136, 171, 146])


In [18]:
from monai.networks.nets import UNet

model = UNet(

    spatial_dims=3,

    in_channels=4,

    out_channels=4, # Changed from 3 to 4 to match 4 classes (0,1,2,3)

    channels=(16,32,64,128,256),

    strides=(2,2,2,2),

    num_res_units=2,

)

In [19]:
import torch
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import Activations, AsDiscrete
from monai.data import decollate_batch
import os

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set CUDA_LAUNCH_BLOCKING for more precise error messages
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Print model's out_channels before moving to device for debugging
if hasattr(model, 'out_channels'):
    print(f"Model out_channels before moving to device: {model.out_channels}")
else:
    print("Model does not have an out_channels attribute or is not a MONAI UNet")

model = model.to(device)

# Dice Loss for segmentation
loss_function = DiceLoss(to_onehot_y=True, softmax=True)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Dice Metric
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Post-processing
post_pred = Compose([
    Activations(softmax=True),
    AsDiscrete(argmax=True, to_onehot=4) # Changed from 3 to 4
])

post_label = Compose([
    AsDiscrete(to_onehot=4) # Changed from 3 to 4
])

print(f"Using device: {device}")

Model out_channels before moving to device: 4
Using device: cuda


In [20]:
# Use only first 10 patients for testing
small_train_files = train_files[:10]

train_ds = Dataset(
    data=small_train_files,
    transform=train_transforms
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True
)

print(f"Training samples: {len(train_ds)}")

Training samples: 10


In [21]:
import torch.nn.functional as F

max_epochs = 3  # Keep small for testing

for epoch in range(max_epochs):

    print(f"\nEpoch {epoch+1}/{max_epochs}")

    model.train()

    epoch_loss = 0

    for batch_data in train_loader:

        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        # Calculate padding needed
        current_d, current_h, current_w = inputs.shape[2:]
        target_d = current_d + (16 - current_d % 16) % 16
        target_h = current_h + (16 - current_h % 16) % 16
        target_w = current_w + (16 - current_w % 16) % 16

        pad_d_end = target_d - current_d
        pad_h_end = target_h - current_h
        pad_w_end = target_w - current_w

        # F.pad expects padding in reverse order for last dimensions: (W_start, W_end, H_start, H_end, D_start, D_end)
        padding = (0, pad_w_end, 0, pad_h_end, 0, pad_d_end)

        # Pad inputs and labels to make dimensions divisible by 16
        inputs = F.pad(inputs, padding, "constant", 0)
        labels = F.pad(labels, padding, "constant", 0)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_function(outputs, labels)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    print(f"Average Loss: {avg_loss:.4f}")


Epoch 1/3
Average Loss: 0.8745

Epoch 2/3
Average Loss: 0.8642

Epoch 3/3
Average Loss: 0.8557


In [22]:
# Re-running training loop
pass

In [24]:
import torch.nn.functional as F

model.eval()

with torch.no_grad():

    for batch_data in train_loader:

        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        # Calculate padding needed (same as in training loop)
        current_d, current_h, current_w = inputs.shape[2:]
        target_d = current_d + (16 - current_d % 16) % 16
        target_h = current_h + (16 - current_h % 16) % 16
        target_w = current_w + (16 - current_w % 16) % 16

        pad_d_end = target_d - current_d
        pad_h_end = target_h - current_h
        pad_w_end = target_w - current_w

        # F.pad expects padding in reverse order for last dimensions: (W_start, W_end, H_start, H_end, D_start, D_end)
        padding = (0, pad_w_end, 0, pad_h_end, 0, pad_d_end)

        # Pad inputs to make dimensions divisible by 16
        inputs = F.pad(inputs, padding, "constant", 0)
        # Labels also need to be padded to match outputs during metric calculation if needed,
        # but for evaluation output, it's inputs that caused the issue.
        labels = F.pad(labels, padding, "constant", 0) # Pad labels too for consistent shape with outputs

        outputs = model(inputs)

        # Convert predictions
        outputs = [post_pred(i) for i in decollate_batch(outputs)]
        labels = [post_label(i) for i in decollate_batch(labels)]

        # Update metric
        dice_metric(y_pred=outputs, y=labels)

    mean_dice = dice_metric.aggregate().item()
    dice_metric.reset()

print(f"\nDice Score (Segmentation Accuracy): {mean_dice:.4f}")


Dice Score (Segmentation Accuracy): 0.0395


### Improving Model Performance: Training with Validation and More Epochs

To address the low Dice score, we will now:
1.  Split the `train_files` (which contains all patients) into a dedicated training set and a validation set.
2.  Define a separate set of transforms for the validation data.
3.  Create new `DataLoader` instances for both training and validation.
4.  Implement a new training loop that runs for more epochs and evaluates the model on the validation set after each epoch. We will save the model that achieves the best Dice score on the validation set.

In [32]:
import math

# Define the split ratio
val_ratio = 0.2
num_total = len(train_files)
# Reduce the number of training patients significantly for faster epochs
num_train = 20 # Changed from int(math.ceil(num_total * (1 - val_ratio))) to a small fixed number
num_val = int(math.ceil(num_total * val_ratio))

# Split the full train_files into training and validation sets
train_files_full = train_files[:num_train]
val_files = train_files[num_total - num_val:] # Ensure validation samples are distinct from training samples, from the end of the list

print(f"Total data samples: {num_total}")
print(f"Training samples: {len(train_files_full)}")
print(f"Validation samples: {len(val_files)}")

Total data samples: 1251
Training samples: 20
Validation samples: 251


In [34]:
from monai.transforms import RandFlipd, RandAffined, RandSpatialCropd

# Validation transforms (typically less augmentation, or just deterministic ones)
val_transforms = Compose([

    LoadImaged(keys=["image","label"]),

    EnsureChannelFirstd(keys=["image","label"]),

    MapLabelValued(
        keys="label",
        orig_labels=(1, 2, 4),
        target_labels=(1, 2, 3)
    ),

    Lambdad(
        keys="label",
        func=lambda x: x.clamp(min=0, max=3).int()
    ),

    Orientationd(
        keys=["image","label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image","label"],
        pixdim=(1.0,1.0,1.0),
        mode=("bilinear","nearest")
    ),

    ScaleIntensityRanged(

        keys=["image"],

        a_min=0,
        a_max=3000,

        b_min=0,
        b_max=1,

        clip=True

    ),

    CropForegroundd(
        keys=["image","label"],
        source_key="image"
    ),

    ToTensord(
        keys=["image","label"]
    )

])

# Augmentations for training data (more robust training)
train_transforms_aug = Compose([

    LoadImaged(keys=["image","label"]),

    EnsureChannelFirstd(keys=["image","label"]),

    MapLabelValued(
        keys="label",
        orig_labels=(1, 2, 4),
        target_labels=(1, 2, 3)
    ),

    Lambdad(
        keys="label",
        func=lambda x: x.clamp(min=0, max=3).int()
    ),

    Orientationd(
        keys=["image","label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image","label"],
        pixdim=(1.0,1.0,1.0),
        mode=("bilinear","nearest")
    ),

    ScaleIntensityRanged(

        keys=["image"],

        a_min=0,
        a_max=3000,

        b_min=0,
        b_max=1,

        clip=True

    ),

    CropForegroundd(
        keys=["image","label"],
        source_key="image"
    ),

    # Add some common augmentations
    RandSpatialCropd(keys=["image", "label"], roi_size=[96, 96, 96], random_size=False, max_roi_size=None, random_center=True),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.10),
    RandAffined(keys=["image", "label"], mode=("bilinear", "nearest"), prob=0.2, spatial_size=[128, 128, 128],
                rotate_range=(math.pi/12, math.pi/12, math.pi/12), scale_range=(0.1, 0.1, 0.1)),

    ToTensord(
        keys=["image","label"]
    )

])

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [35]:
# Create Dataset and DataLoader for the full training and validation sets
train_ds_full = Dataset(
    data=train_files_full,
    transform=train_transforms_aug
)

train_loader_full = DataLoader(
    train_ds_full,
    batch_size=1,
    shuffle=True
)

val_ds = Dataset(
    data=val_files,
    transform=val_transforms
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False # No need to shuffle validation data
)

print(f"New training data loader size: {len(train_loader_full)}")
print(f"Validation data loader size: {len(val_loader)}")

New training data loader size: 20
Validation data loader size: 251


In [36]:
import torch.nn.functional as F
import numpy as np

# Re-initialize the model to ensure a fresh start if any previous training occurred
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16,32,64,128,256),
    strides=(2,2,2,2),
    num_res_units=2,
).to(device)

# Re-initialize optimizer (if using model_reinitialize, otherwise use existing one)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Define a higher number of epochs for proper training
max_epochs_new = 20  # Increased epochs

# To keep track of the best model
best_metric = -1
best_metric_epoch = -1

print(f"Starting training for {max_epochs_new} epochs...")

for epoch in range(max_epochs_new):

    print(f"\nEpoch {epoch+1}/{max_epochs_new}")

    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader_full:

        step += 1
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        # Dynamic padding (same as before)
        current_d, current_h, current_w = inputs.shape[2:]
        target_d = current_d + (16 - current_d % 16) % 16
        target_h = current_h + (16 - current_h % 16) % 16
        target_w = current_w + (16 - current_w % 16) % 16

        pad_d_end = target_d - current_d
        pad_h_end = target_h - current_h
        pad_w_end = target_w - current_w

        padding = (0, pad_w_end, 0, pad_h_end, 0, pad_d_end)

        inputs = F.pad(inputs, padding, "constant", 0)
        labels = F.pad(labels, padding, "constant", 0)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        # print(f"{step}/{len(train_loader_full)}, train_loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader_full)
    print(f"Average Training Loss: {avg_loss:.4f}")

    # Validation phase
    if (epoch + 1) % 5 == 0: # Evaluate every 5 epochs, or adjust as needed
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                inputs_val = val_data["image"].to(device)
                labels_val = val_data["label"].to(device)

                # Dynamic padding for validation (same as training)
                current_d, current_h, current_w = inputs_val.shape[2:]
                target_d = current_d + (16 - current_d % 16) % 16
                target_h = current_h + (16 - current_h % 16) % 16
                target_w = current_w + (16 - current_w % 16) % 16

                pad_d_end = target_d - current_d
                pad_h_end = target_h - current_h
                pad_w_end = target_w - current_w

                padding_val = (0, pad_w_end, 0, pad_h_end, 0, pad_d_end)

                inputs_val = F.pad(inputs_val, padding_val, "constant", 0)
                labels_val = F.pad(labels_val, padding_val, "constant", 0)

                outputs_val = model(inputs_val)

                outputs_val = [post_pred(i) for i in decollate_batch(outputs_val)]
                labels_val = [post_label(i) for i in decollate_batch(labels_val)]

                dice_metric(y_pred=outputs_val, y=labels_val)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()

            print(f"Validation Dice Score: {metric:.4f}")

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_metric_model.pth")
                print("Saved new best metric model!")

print(f"\nTraining finished. Best validation Dice score: {best_metric:.4f} at epoch {best_metric_epoch}")

Starting training for 20 epochs...

Epoch 1/20
Average Training Loss: 0.8580

Epoch 2/20
Average Training Loss: 0.8423

Epoch 3/20
Average Training Loss: 0.8316

Epoch 4/20
Average Training Loss: 0.8096

Epoch 5/20
Average Training Loss: 0.8036
Validation Dice Score: 0.0731
Saved new best metric model!

Epoch 6/20
Average Training Loss: 0.7902

Epoch 7/20
Average Training Loss: 0.7950

Epoch 8/20
Average Training Loss: 0.7709

Epoch 9/20
Average Training Loss: 0.7604

Epoch 10/20
Average Training Loss: 0.7554
Validation Dice Score: 0.0800
Saved new best metric model!

Epoch 11/20
Average Training Loss: 0.7553

Epoch 12/20
Average Training Loss: 0.7430

Epoch 13/20
Average Training Loss: 0.7326

Epoch 14/20
Average Training Loss: 0.7398

Epoch 15/20
Average Training Loss: 0.7113
Validation Dice Score: 0.0943
Saved new best metric model!

Epoch 16/20
Average Training Loss: 0.7168

Epoch 17/20
Average Training Loss: 0.7118

Epoch 18/20
Average Training Loss: 0.6947

Epoch 19/20
Average Tr

In [37]:
# Running evaluation
pass

In [38]:
torch.save(model.state_dict(), "baseline_unet.pth")

print("Baseline model saved!")

Baseline model saved!


In [39]:
# Saving the model
pass

In [40]:
#from google.colab import files
#files.download("baseline_unet.pth")

In [41]:
# Downloading the model
pass